# Teste do Qwen3-8B ajustado para Visual FoxPro 9

Avalia o adaptador salvo em `final_model` de três formas: perguntas de conhecimento, um laço
agêntico real contra um projeto VFP sintético e um placar sobre várias tarefas de correção.

O laço é de verdade. O modelo emite `<tool_call>`, o notebook interpreta a chamada, executa a
ferramenta sobre arquivos em memória e devolve `<tool_response>`. Nada de resposta pré-gravada:
se o modelo pedir uma linha errada ou inventar um nome de ferramenta, o erro aparece.

## O que você precisa subir

Um arquivo só: **`data/agentic/projeto_teste.json`** (400 KB), que está no repositório. Coloque-o
no Drive em `VFP-LLM-Qwen3-8B-2/testing/` ou simplesmente arraste para `/content` no painel
*Arquivos* — a célula 4 procura nos dois lugares.

Ele contém o projeto sintético (11 arquivos, 219 rotinas), os esquemas das quatro ferramentas, o
prompt de sistema do agente e 442 defeitos catalogados, cada um marcado como visto ou não durante
o treino.

O resto já está no Drive, salvo pelo notebook de treino: `final_model/` com o adaptador de 166 MB e
o tokenizer. O modelo base vem do Hub.

## 1. Google Drive, cache e diretórios

In [ ]:
import os

from google.colab import drive

# ============================================================
# GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/VFP-LLM-Qwen3-8B-2"

# ============================================================
# CACHE DA HUGGING FACE
# ============================================================

HF_HOME = os.path.join(
    PROJECT_DIR,
    "huggingface",
)

os.environ["HF_HOME"] = HF_HOME
os.environ["HF_HUB_CACHE"] = os.path.join(HF_HOME, "hub")
os.environ["HF_XET_CACHE"] = os.path.join(HF_HOME, "xet")
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_HOME, "datasets")

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ============================================================
# DIRETÓRIOS
# ============================================================

FINAL_MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "final_model",
)

TESTING_DIR = os.path.join(
    PROJECT_DIR,
    "testing",
)

os.makedirs(
    TESTING_DIR,
    exist_ok=True,
)

os.chdir(PROJECT_DIR)

if not os.path.isdir(FINAL_MODEL_DIR):
    raise FileNotFoundError(
        f"{FINAL_MODEL_DIR} não existe. Rode antes a célula 14 do notebook de treino."
    )

print("Projeto:", PROJECT_DIR)
print("Adaptador:", FINAL_MODEL_DIR)

## 2. Dependências

Se o Colab pedir para reiniciar a sessão, reinicie e recomece da célula 1.

In [ ]:
%pip install -q -U \
    "transformers>=4.51" \
    "accelerate>=1.0" \
    "peft>=0.14" \
    "bitsandbytes>=0.45"

## 3. Imports e hardware

In [ ]:
import json
import re
import shutil
import textwrap

import torch
import transformers

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

if not torch.cuda.is_available():
    raise RuntimeError("Sem GPU. Ambiente de execução -> Alterar o tipo -> GPU.")

USA_BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USA_BF16 else torch.float16

print("transformers:", transformers.__version__)
print("GPU.........:", torch.cuda.get_device_name(0))
print("precisão....:", "bf16" if USA_BF16 else "fp16")

## 4. Localizar o projeto de teste

Procura `projeto_teste.json` no Drive e em `/content`, e copia para o Drive quando o acha fora —
`/content` some quando a sessão cai.

In [ ]:
NOME_PROJETO = "projeto_teste.json"

ORIGENS = [
    TESTING_DIR,
    "/content",
    os.getcwd(),
    "/content/drive/MyDrive",
]

origem = next(
    (os.path.join(pasta, NOME_PROJETO) for pasta in ORIGENS
     if os.path.isfile(os.path.join(pasta, NOME_PROJETO))),
    None,
)

if origem is None:
    from google.colab import files

    print(f"Não encontrei {NOME_PROJETO}. Selecione o arquivo abaixo.\n")
    enviados = files.upload()

    if NOME_PROJETO not in enviados:
        raise FileNotFoundError(f"{NOME_PROJETO} não foi enviado.")

    origem = os.path.join(TESTING_DIR, NOME_PROJETO)

    with open(origem, "wb") as arquivo:
        arquivo.write(enviados[NOME_PROJETO])

CAMINHO_PROJETO = os.path.join(TESTING_DIR, NOME_PROJETO)

if os.path.abspath(origem) != os.path.abspath(CAMINHO_PROJETO):
    shutil.copyfile(origem, CAMINHO_PROJETO)
    print(f"copiado de {origem}")

with open(CAMINHO_PROJETO, encoding="utf-8") as arquivo:
    PACOTE = json.load(arquivo)

PROMPT_SISTEMA = PACOTE["prompt_sistema"]
FERRAMENTAS = PACOTE["ferramentas"]
DEFEITOS = PACOTE["defeitos"]
FICHAS = PACOTE["fichas"]

INEDITOS = [d for d in DEFEITOS if not d["visto_no_treino"]]

print(f"arquivos.............: {len(PACOTE['arquivos'])}")
print(f"rotinas..............: {len(PACOTE['rotinas'])}")
print(f"ferramentas..........: {[f['function']['name'] for f in FERRAMENTAS]}")
print(f"defeitos catalogados.: {len(DEFEITOS)} ({len(INEDITOS)} nunca vistos no treino)")
print(f"fichas de referência.: {len(FICHAS)}")

## 5. Carregar o modelo base e o adaptador

O base entra em 4 bits, como no treino, e o adaptador de 166 MB é aplicado por cima. O tokenizer
vem de `final_model`, para usar exatamente o mesmo template de chat do treino.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"

quantizacao = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)

chave_dtype = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantizacao,
    attn_implementation="sdpa",
    device_map={"": 0},
    trust_remote_code=True,
    **{chave_dtype: DTYPE},
)

modelo = PeftModel.from_pretrained(base, FINAL_MODEL_DIR)
modelo.eval()
modelo.config.use_cache = True

tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR, trust_remote_code=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"memória ocupada: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 6. As quatro ferramentas

Reimplementadas sobre os arquivos em memória, com os mesmos formatos de saída usados no treino: a
numeração de `read_file`, o `arquivo:linha: conteúdo` de `search_text` e o `ok: 1 substituição em`
de `edit_file`. Qualquer diferença aqui viraria distribuição fora do treino e penalizaria o modelo
por um motivo que não é dele.

O `edit_file` exige que `old_text` ocorra **uma única vez**. Quando não ocorre, devolve um erro —
situação que o modelo nunca viu no treino, já que todos os traces mostram sucesso. É de propósito:
serve para observar se ele tenta se recuperar ou trava.

O `search_text` merece um aviso. Comparei estas implementações com as 2.296 respostas gravadas nos
traces: `list_files`, `read_file` e `edit_file` batem em 100%, mas a busca não bate — e o motivo é
que boa parte das respostas de busca do treino era **fabricada**, não produzida por uma busca real.
Em 166 traces o modelo procura `FUNCTION P_Algo` e recebe de volta uma linha com `PROCEDURE P_Algo`,
que nenhum grep honesto encontraria. Em outros 152 a resposta é "Nenhuma ocorrência encontrada."
para rotinas que existem no projeto.

Aqui a busca é real, e isso é proposital: se o agente só funciona quando a ferramenta mente para
ele, é melhor descobrir agora. O único ajuste que fiz foi o `CONVENCAO_DO_TREINO`, que reporta a
linha inicial do bloco da rotina em vez da linha do `FUNCTION` — a convenção da maioria dos traces,
uma linha acima. Sem ele, o modelo lê o intervalo deslocado em uma linha e pode perder o asterisco
de abertura do cabeçalho. Troque para `False` para medir essa sensibilidade.

In [ ]:
CONVENCAO_DO_TREINO = True

CABECALHO_RE = re.compile(r"^\s*(?:FUNCTION|PROCEDURE)\s+(\S+)", re.I)


class Projeto:
    """Cópia de trabalho do projeto sintético, com as ferramentas do agente."""

    def __init__(self, pacote: dict) -> None:
        self.arquivos = dict(pacote["arquivos"])
        self.rotinas = pacote["rotinas"]

    def copia(self) -> "Projeto":
        return Projeto({"arquivos": self.arquivos, "rotinas": self.rotinas})

    # -- ferramentas -------------------------------------------------------- #
    def list_files(self) -> str:
        linhas = []

        for caminho in sorted(self.arquivos):
            total = len(self.arquivos[caminho].split("\n"))
            rotinas = sum(1 for r in self.rotinas.values() if r["arquivo"] == caminho)
            linhas.append(f"{caminho}  ({total} linhas, {rotinas} rotinas)")

        return "\n".join(linhas)

    def _linha_relatada(self, caminho: str, numero: int, linha: str) -> int:
        """Na convenção do treino, um cabeçalho de rotina reporta o início do bloco."""
        if not CONVENCAO_DO_TREINO:
            return numero

        cabecalho = CABECALHO_RE.match(linha)

        if not cabecalho:
            return numero

        rotina = self.rotinas.get(cabecalho.group(1))

        if rotina and rotina["arquivo"] == caminho:
            return rotina["linha_inicial"]

        return numero

    def search_text(self, pattern: str, limite: int = 12) -> str:
        try:
            regex = re.compile(pattern, re.I)
        except re.error as erro:
            return f"erro: padrão inválido ({erro})"

        achados = []

        for caminho in sorted(self.arquivos):
            for numero, linha in enumerate(self.arquivos[caminho].split("\n"), 1):
                if regex.search(linha):
                    relatada = self._linha_relatada(caminho, numero, linha)
                    achados.append(f"{caminho}:{relatada}: {linha.strip()}")

                    if len(achados) >= limite:
                        return "\n".join(achados)

        return "\n".join(achados) if achados else "Nenhuma ocorrência encontrada."

    def read_file(self, path: str, start_line: int, end_line: int) -> str:
        if path not in self.arquivos:
            return f"erro: arquivo {path} não existe"

        linhas = self.arquivos[path].split("\n")
        inicio = max(1, int(start_line))
        fim = min(len(linhas), int(end_line))

        if inicio > fim:
            return "erro: intervalo vazio"

        return "\n".join(f"{n:5d}| {linhas[n - 1]}" for n in range(inicio, fim + 1))

    def edit_file(self, path: str, old_text: str, new_text: str) -> str:
        if path not in self.arquivos:
            return f"erro: arquivo {path} não existe"

        ocorrencias = self.arquivos[path].count(old_text)

        if ocorrencias == 0:
            return f"erro: o trecho não foi encontrado em {path}"

        if ocorrencias > 1:
            return f"erro: o trecho ocorre {ocorrencias} vezes em {path}; use um trecho maior"

        self.arquivos[path] = self.arquivos[path].replace(old_text, new_text, 1)

        return f"ok: 1 substituição em {path}"


ESQUEMAS = {
    ferramenta["function"]["name"]: ferramenta["function"]["parameters"]
    for ferramenta in FERRAMENTAS
}


def executar(projeto: Projeto, nome: str, argumentos: dict) -> tuple[str, str | None]:
    """Roda a ferramenta e devolve (resposta, falha_de_protocolo)."""
    if nome not in ESQUEMAS:
        return f"erro: ferramenta {nome} não existe", "nome de ferramenta inválido"

    obrigatorios = set(ESQUEMAS[nome].get("required", []))
    faltando = obrigatorios - set(argumentos)

    if faltando:
        return f"erro: faltam os argumentos {sorted(faltando)}", "argumento obrigatório ausente"

    try:
        return getattr(projeto, nome)(**argumentos), None
    except TypeError as erro:
        return f"erro: {erro}", "argumentos fora do esquema"


projeto_base = Projeto(PACOTE)

print(projeto_base.list_files())

## 7. O laço do agente

Gera, procura um `<tool_call>`, executa e devolve o resultado como mensagem de papel `tool` — a
mesma estrutura do treino. Repete até o modelo responder em texto puro ou até o limite de passos.

A geração é determinística (`do_sample=False`) para que rodar duas vezes dê o mesmo resultado.

In [ ]:
CHAMADA_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)


def gerar(
    mensagens: list[dict],
    ferramentas: list[dict] | None = FERRAMENTAS,
    maximo: int = 512,
) -> str:
    # Passar ferramentas=None é essencial nas perguntas de conhecimento: o template
    # do Qwen injeta a lista de ferramentas no prompt de sistema, e a simples
    # presença dela faz o modelo entrar em modo agente e chamar search_text.
    codificado = tokenizer.apply_chat_template(
        mensagens,
        tools=ferramentas,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    # Conforme a versão do transformers isto vem como tensor puro ou como
    # BatchEncoding; return_dict passou a ser True por padrão na série 5.
    if torch.is_tensor(codificado):
        codificado = {"input_ids": codificado}

    entrada = {chave: valor.to(modelo.device) for chave, valor in codificado.items()}
    tamanho = entrada["input_ids"].shape[-1]

    with torch.no_grad():
        saida = modelo.generate(
            **entrada,
            max_new_tokens=maximo,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    texto = tokenizer.decode(saida[0][tamanho:], skip_special_tokens=False)

    return texto.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()


def extrair_chamada(texto: str) -> tuple[dict | None, str | None]:
    """Devolve (chamada, falha). Sem <tool_call>, ambos None significa resposta final."""
    if "<tool_call>" not in texto:
        return None, None

    achado = CHAMADA_RE.search(texto)

    if achado is None:
        return None, "bloco <tool_call> sem fechamento ou sem JSON"

    try:
        dados = json.loads(achado.group(1))
    except json.JSONDecodeError as erro:
        return None, f"JSON inválido na chamada ({erro})"

    if "name" not in dados:
        return None, "chamada sem o campo name"

    argumentos = dados.get("arguments", {})

    if isinstance(argumentos, str):
        try:
            argumentos = json.loads(argumentos)
        except json.JSONDecodeError:
            return None, "arguments não é um objeto JSON"

    return {"name": dados["name"], "arguments": argumentos}, None


def rodar_agente(tarefa: str, projeto: Projeto, max_passos: int = 8, verboso: bool = True) -> dict:
    mensagens = [
        {"role": "system", "content": PROMPT_SISTEMA},
        {"role": "user", "content": tarefa},
    ]

    falhas: list[str] = []
    ferramentas_usadas: list[str] = []
    resposta_final = None

    for passo in range(1, max_passos + 1):
        texto = gerar(mensagens)
        chamada, falha = extrair_chamada(texto)

        if falha:
            falhas.append(falha)

            if verboso:
                print(f"[{passo}] FALHA DE PROTOCOLO: {falha}\n{texto[:400]}\n")

            break

        if chamada is None:
            resposta_final = texto
            mensagens.append({"role": "assistant", "content": texto})

            if verboso:
                print(f"[{passo}] resposta final:\n{textwrap.indent(texto, '    ')}\n")

            break

        resultado, falha_execucao = executar(projeto, chamada["name"], chamada["arguments"])
        ferramentas_usadas.append(chamada["name"])

        if falha_execucao:
            falhas.append(falha_execucao)

        if resultado.startswith("erro:"):
            falhas.append(f"ferramenta devolveu erro: {resultado}")

        if verboso:
            argumentos = json.dumps(chamada["arguments"], ensure_ascii=False)
            print(f"[{passo}] {chamada['name']}({argumentos[:160]})")
            print(textwrap.indent(resultado[:500], "    | "))
            print()

        mensagens.append(
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": f"call_{passo}",
                        "type": "function",
                        "function": {"name": chamada["name"], "arguments": chamada["arguments"]},
                    }
                ],
            }
        )
        mensagens.append(
            {"role": "tool", "tool_call_id": f"call_{passo}", "content": resultado}
        )
    else:
        falhas.append(f"não concluiu em {max_passos} passos")

    return {
        "mensagens": mensagens,
        "ferramentas": ferramentas_usadas,
        "falhas": falhas,
        "resposta": resposta_final,
    }


print("laço pronto")

## 8. Conhecimento

Sem ferramentas: aqui só interessa se o modelo responde em português, sobre VFP9, e sem tentar
chamar ninguém. Um `<tool_call>` nestas respostas seria sinal de que o prompt de assistente e o de
agente se confundiram no treino.

In [ ]:
PROMPT_ASSISTENTE = (
    "Você é um assistente especializado em Microsoft Visual FoxPro 9. "
    "Responda de forma técnica e objetiva, em português."
)

PERGUNTAS = [
    "Qual a diferença entre SEEK e LOCATE?",
    "Para que serve a função SYS(2015)?",
    "Como declaro um array de duas dimensões e percorro seus elementos?",
    "O que faz o comando SET DELETED ON?",
    "Qual a sintaxe da função STRTRAN e o que cada parâmetro significa?",
]

# Comparar duas listas de palavras funcionais é mais confiável do que contar
# ocorrências de uma só: respostas curtas raramente têm três palavras da lista.
PALAVRAS_PT = {"a", "o", "as", "os", "de", "da", "do", "que", "para", "com", "uma", "um",
               "não", "em", "por", "na", "no", "se", "são", "como", "é", "ser", "pode",
               "cada", "após", "ocorrências", "expressão", "caracteres"}
PALAVRAS_EN = {"the", "of", "and", "to", "in", "is", "for", "that", "with", "you", "are",
               "this", "it", "as", "be", "or", "from", "which", "specifies", "returns"}

for pergunta in PERGUNTAS:
    resposta = gerar(
        [
            {"role": "system", "content": PROMPT_ASSISTENTE},
            {"role": "user", "content": pergunta},
        ],
        ferramentas=None,
        maximo=320,
    )

    palavras = re.findall(r"[a-zà-ú]+", resposta.lower())
    pontos_pt = sum(1 for p in palavras if p in PALAVRAS_PT)
    pontos_en = sum(1 for p in palavras if p in PALAVRAS_EN)
    parece_pt = pontos_pt > pontos_en
    chamou = "<tool_call>" in resposta

    print("=" * 78)
    print("P:", pergunta)
    print(f"   português: {'sim' if parece_pt else 'NÃO'} | chamou ferramenta: {'SIM' if chamou else 'não'}")
    print("=" * 78)
    print(textwrap.indent(resposta[:900], "  "))
    print()

## 8b. Fidelidade das assinaturas

O teste anterior mostra se o modelo responde; este mede se ele acerta. Sorteia funções cuja ficha
esteve no treino, pede a assinatura e compara com a referência.

É a pergunta central sobre injetar conhecimento nos pesos: o modelo aprendeu os fatos ou só o
formato em que os fatos eram apresentados? Uma assinatura com o formato impecável e nomes de
parâmetro inventados é o pior resultado possível, porque parece certa.

In [ ]:
import random

AMOSTRA = 30

BLOCO_FOXPRO_RE = re.compile(r"```foxpro\n(.*?)\n```", re.S)
IDENTIFICADOR_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")


def parametros(assinatura: str) -> list[str]:
    """Nomes dos parâmetros, na ordem, ignorando colchetes e separadores."""
    abre = assinatura.find("(")
    fecha = assinatura.rfind(")")

    if abre < 0 or fecha < abre:
        return []

    return IDENTIFICADOR_RE.findall(assinatura[abre + 1 : fecha])


sorteadas = random.Random(42).sample(FICHAS, min(AMOSTRA, len(FICHAS)))

placar = {"exata": 0, "só os parâmetros": 0, "errada": 0, "sem bloco de código": 0}
divergentes = []

for ficha in sorteadas:
    resposta = gerar(
        [
            {"role": "system", "content": PROMPT_ASSISTENTE},
            {"role": "user", "content": f"Qual a assinatura de {ficha['nome']}( ) no Visual FoxPro 9?"},
        ],
        ferramentas=None,
        maximo=220,
    )

    bloco = BLOCO_FOXPRO_RE.search(resposta)

    if bloco is None:
        placar["sem bloco de código"] += 1
        continue

    obtida = " ".join(bloco.group(1).split())
    esperada = " ".join(ficha["assinatura"].split())

    if obtida == esperada:
        placar["exata"] += 1
    elif parametros(obtida) == parametros(esperada):
        placar["só os parâmetros"] += 1
    else:
        placar["errada"] += 1
        divergentes.append((ficha["nome"], esperada, obtida))

total = len(sorteadas)

print(f"assinaturas conferidas: {total}\n")

for rotulo, quantidade in placar.items():
    print(f"  {rotulo:22s} {quantidade:3d}  ({quantidade / total:5.1%})")

if divergentes:
    print()
    print("=" * 78)
    print("DIVERGÊNCIAS")
    print("=" * 78)

    for nome, esperada, obtida in divergentes[:8]:
        print(f"\n{nome}")
        print(f"  referência: {esperada[:120]}")
        print(f"  modelo....: {obtida[:120]}")

## 9. Um trace agêntico completo

Injeta num arquivo do projeto um defeito que o modelo **nunca viu no treino** e pede a correção.
Cada passo aparece: a chamada que ele fez e o que a ferramenta devolveu.

No fim, a verificação que importa: o arquivo voltou ao código correto?

In [ ]:
def preparar_defeito(defeito: dict) -> tuple[Projeto, str]:
    """Copia o projeto e troca a rotina correta pela versão quebrada."""
    projeto = projeto_base.copia()
    caminho = defeito["arquivo"]
    texto = projeto.arquivos[caminho]

    if defeito["codigo_correto"] not in texto:
        raise RuntimeError(f"não localizei {defeito['rotina']} em {caminho}")

    projeto.arquivos[caminho] = texto.replace(
        defeito["codigo_correto"], defeito["codigo_quebrado"], 1
    )

    return projeto, caminho


escolhido = INEDITOS[0] if INEDITOS else DEFEITOS[0]
projeto_teste, arquivo_alvo = preparar_defeito(escolhido)
texto_quebrado = projeto_teste.arquivos[arquivo_alvo]

print(f"rotina : {escolhido['rotina']}  ({arquivo_alvo})")
print(f"defeito: {escolhido['erro']}")
print(f"inédito: {'sim' if not escolhido['visto_no_treino'] else 'não'}")
print("=" * 78)
print()

execucao = rodar_agente(
    f"A rotina {escolhido['rotina']} não está se comportando como deveria. Corrija.",
    projeto_teste,
)

restaurou = escolhido["codigo_correto"] in projeto_teste.arquivos[arquivo_alvo]
mudou = projeto_teste.arquivos[arquivo_alvo] != texto_quebrado

print("=" * 78)
print(f"ferramentas usadas.: {' -> '.join(execucao['ferramentas']) or 'nenhuma'}")
print(f"falhas de protocolo: {execucao['falhas'] or 'nenhuma'}")
print(f"alterou o arquivo..: {'sim' if mudou else 'NÃO'}")
print(f"restaurou o correto: {'sim' if restaurou else 'NÃO'}")

## 10. Placar

Roda várias tarefas e resume. Três famílias:

- **corrigir** — defeitos injetados, priorizando os inéditos;
- **localizar** — perguntas de "onde está X", que exigem `search_text` e nenhuma edição;
- **inspecionar** — perguntas de "quais rotinas fazem X", que devem terminar em texto.

Repare na coluna de edições indevidas: um agente que altera arquivo numa tarefa de consulta é pior
do que um que erra a resposta, porque o estrago é silencioso.

In [ ]:
def tarefas_de_correcao(quantidade: int = 4) -> list[dict]:
    escolhidos = list(INEDITOS)

    for defeito in DEFEITOS:
        if len(escolhidos) >= quantidade:
            break

        if defeito not in escolhidos:
            escolhidos.append(defeito)

    return escolhidos[:quantidade]


resultados = []

for defeito in tarefas_de_correcao():
    projeto, caminho = preparar_defeito(defeito)
    quebrado = projeto.arquivos[caminho]

    execucao = rodar_agente(
        f"A rotina {defeito['rotina']} não está se comportando como deveria. Corrija.",
        projeto,
        verboso=False,
    )

    resultados.append(
        {
            "tarefa": f"corrigir {defeito['rotina']}",
            "inédito": not defeito["visto_no_treino"],
            "passos": len(execucao["ferramentas"]),
            "falhas": len(execucao["falhas"]),
            "sucesso": defeito["codigo_correto"] in projeto.arquivos[caminho],
            "editou": projeto.arquivos[caminho] != quebrado,
            "detalhe": execucao["falhas"][0] if execucao["falhas"] else "",
        }
    )

NOMES_ROTINAS = sorted(PACOTE["rotinas"])

CONSULTAS = [
    ("localizar", f"Em que arquivo está a rotina {NOMES_ROTINAS[10]}?"),
    ("localizar", f"Onde fica {NOMES_ROTINAS[120]}? Só me diga o arquivo e a linha."),
    ("inspecionar", "Quais rotinas de manipulação de texto existem no projeto?"),
    ("inspecionar", "Liste as rotinas que trabalham com datas."),
]

for familia, pergunta in CONSULTAS:
    projeto = projeto_base.copia()
    antes = dict(projeto.arquivos)

    execucao = rodar_agente(pergunta, projeto, verboso=False)

    resultados.append(
        {
            "tarefa": f"{familia}: {pergunta[:34]}...",
            "inédito": True,
            "passos": len(execucao["ferramentas"]),
            "falhas": len(execucao["falhas"]),
            "sucesso": execucao["resposta"] is not None,
            "editou": projeto.arquivos != antes,
            "detalhe": execucao["falhas"][0] if execucao["falhas"] else "",
        }
    )

print(f"{'tarefa':46s} {'inéd':>5s} {'passos':>7s} {'falhas':>7s} {'ok':>4s} {'editou':>7s}")
print("-" * 82)

for linha in resultados:
    print(
        f"{linha['tarefa'][:46]:46s} "
        f"{'sim' if linha['inédito'] else 'não':>5s} "
        f"{linha['passos']:>7d} {linha['falhas']:>7d} "
        f"{'sim' if linha['sucesso'] else 'NÃO':>4s} "
        f"{'sim' if linha['editou'] else 'não':>7s}"
    )

print("-" * 82)

total = len(resultados)
limpos = sum(1 for r in resultados if r["falhas"] == 0)
ok = sum(1 for r in resultados if r["sucesso"])
indevidas = sum(1 for r in resultados if r["editou"] and not r["tarefa"].startswith("corrigir"))

print(f"protocolo sem falhas : {limpos}/{total}")
print(f"tarefas concluídas   : {ok}/{total}")
print(f"edições indevidas    : {indevidas}")

tropecos = [(r["tarefa"], r["detalhe"]) for r in resultados if r["detalhe"]]

if tropecos:
    print()
    print("primeira falha de cada tarefa que tropeçou:")

    for tarefa, detalhe in tropecos:
        print(f"  {tarefa[:40]:40s} {detalhe[:120]}")

## 11. Teste livre

Escreva a tarefa e observe o laço. Vale tentar quebrar o agente: peça algo que não existe, peça
duas mudanças de uma vez, ou peça para alterar um arquivo sem dizer qual rotina.

In [ ]:
TAREFA = "Quais rotinas começam com F_Cliente? Só liste os nomes."

# A cópia isola as edições, então esta célula pode rodar quantas vezes você quiser.
execucao = rodar_agente(TAREFA, projeto_base.copia())

print("=" * 78)
print("ferramentas:", " -> ".join(execucao["ferramentas"]) or "nenhuma")
print("falhas.....:", execucao["falhas"] or "nenhuma")